In [0]:
import json

# Get secrets
kafka_connection_json = dbutils.secrets.get(scope='fraudwatch-scope', key='kafka-connection-details')
kafka_config = json.loads(kafka_connection_json)

# Extract values from the json
bootstrap_servers = kafka_config['bootstrap_servers']
topic = kafka_config['topic']
api_key = kafka_config['api_key']
api_secret = kafka_config['api_secret']

In [0]:
jaas_config=f"kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username='{api_key}' password='{api_secret}';"

## Batch Data

In [0]:
sample_batch = (spark.read.format('kafka')
                .option('kafka.bootstrap.servers', bootstrap_servers)
                .option('subscribe', topic)
                .option('kafka.security.protocol', 'SASL_SSL')
                .option('kafka.sasl.mechanism', 'PLAIN')
                .option('kafka.sasl.jaas.config', jaas_config)
                .option('startingOffsets', 'earliest')
                .load()
                )

In [0]:
sample_batch.count()

In [0]:
display(sample_batch)

In [0]:
from pyspark.sql.functions import col

parsed_batch = sample_batch.select(
    col('key').cast('string'),
    col('value').cast('string'),
    col('topic'),
    col('partition'),
    col('offset'),
    col('timestamp'),
    col('timestampType')
)

In [0]:
display(parsed_batch)

In [0]:
# Create a batch table
parsed_batch.write.saveAsTable('fraudwatch.bronze.transactions_batch_test')

## Streaming Data

In [0]:
stream = (
    spark.readStream.format('kafka')
    .option('kafka.bootstrap.servers', bootstrap_servers)
    .option('subscribe', topic)
    .option('kafka.security.protocol', 'SASL_SSL')
    .option('kafka.sasl.mechanism', 'PLAIN')
    .option('kafka.sasl.jaas.config', jaas_config)
    .option('startingOffsets', 'earliest')
    .load()
)

In [0]:
from pyspark.sql.functions import col

parsed_stream = stream.select(
    col('key').cast('string'),
    col('value').cast('string'),
    col('topic'),
    col('partition'),
    col('offset'),
    col('timestamp'),
    col('timestampType')
)

In [0]:
%sql
-- Create a new schema
CREATE SCHEMA IF NOT EXISTS fraudwatch.source;

In [0]:
%sql
-- Create external location for the volume
CREATE EXTERNAL LOCATION IF NOT EXISTS ext_volume
URL 'abfss://data@fraudwatch01.dfs.core.windows.net/volumes'
WITH (STORAGE CREDENTIAL fraudwatch_sc)
COMMENT 'External location for volumes'
;

In [0]:
%sql
-- Create a volume to store the checkpoint
CREATE EXTERNAL VOLUME IF NOT EXISTS fraudwatch.source.transactions
LOCATION 'abfss://data@fraudwatch01.dfs.core.windows.net/volumes'
;

In [0]:
dbutils.fs.help()

In [0]:
%skip
# Create a checkpoints folder in the volume
dbutils.fs.mkdirs('/Volumes/fraudwatch/source/transactions/checkpoints')

In [0]:
stream_query = (parsed_stream.writeStream.format('delta')
    .outputMode('append')
    .option('checkpointLocation', '/Volumes/fraudwatch/source/transactions/checkpoints/')
    .trigger(availableNow=True)
    .toTable('fraudwatch.bronze.transactions_stream_test')
)

print(f"Query started with query id: {stream_query.id}")

In [0]:
display(spark.read.table('fraudwatch.bronze.transactions_stream_test'))